# Task 1: Netflix Data Cleaning & Preparation
**Auspify Technologies, Data Analysis Using Python Internship**

**Objective:** Prepare and organize the Netflix dataset for business analytics and reporting.

**Tools used:** Python, Pandas, NumPy

In [1]:
%pip install pandas matplotlib seaborn plotly

## Step 1: Import Libraries and Load the Dataset
Import the required libraries and load `Dataset.csv` into a Pandas DataFrame.

In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as snsA
import plotly.express as px
print("All libraries loaded")

All libraries loaded


## Step 1 (continued): Load the Dataset
Load `Dataset.csv` into a Pandas DataFrame and preview the first 5 rows.

In [21]:
df = pd.read_csv("Dataset.csv")
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## Understand the Data
Check the number of rows and columns, the column names and the data types.
The dataset has **8,790 rows and 10 columns**.

In [ ]:
print("Shape:", df.shape)
df.info()

## Step 2: Identify Missing Values (Before Cleaning)
Pandas shows 0 missing values, but the dataset uses the text **"Not Given"** as a placeholder.
The hidden missing values are in the `country` and `director` columns.

In [22]:
print(df.isna().sum())
print()
print("'Not Given' in country :", (df["country"] == "Not Given").sum())
print("'Not Given' in director:", (df["director"] == "Not Given").sum())

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

'Not Given' in country : 287
'Not Given' in director: 2588


## Step 2 (continued): Handle Missing Values
Placeholder text is converted to real missing values (NaN) and then filled with **"Unknown"**.
Rows are kept instead of deleted, so no useful data is lost.

In [25]:
import numpy as np

# turn placeholder text into real missing values
df = df.replace(r"^\s*(Not Given|Unknown|)\s*$", np.nan, regex=True)
print(df.isna().sum())

# fill them with a clear label
df["director"] = df["director"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")

print("\nAfter filling:")
print(df.isna().sum())

show_id            0
type               0
title              1
director        2588
country          287
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
dtype: int64

After filling:
show_id         0
type            0
title           1
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64


## Step 3: Remove Duplicates and Fix Formatting
Check for duplicate records, remove them, and strip extra spaces from the text columns.

In [18]:
print("Exact duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

# look at possible duplicates (same title + type + year)
dups = df[df.duplicated(subset=["title", "type", "release_year"], keep=False)]
print(dups)

# remove extra spaces in text columns
for col in ["type", "title", "director", "country", "rating", "listed_in"]:
    df[col] = df[col].str.strip()

Exact duplicates: 0
     show_id     type         title            director   country date_added  \
393    s3997  Tv Show         9-Feb             Unknown  Pakistan 2019-03-20   
537    s5965  Tv Show         9-Feb             Unknown  Pakistan 2019-03-20   
2590   s3372    Movie  Consequences        Ozan Açıktan    Turkey 2019-10-25   
2925   s3963    Movie        15-Aug  Swapnaneel Jayakar     India 2019-03-29   
3285   s4523    Movie        22-Jul     Paul Greengrass    Norway 2018-10-10   
4260   s5966    Movie        22-Jul     Paul Greengrass    Norway 2018-10-10   
4261   s5967    Movie        15-Aug  Swapnaneel Jayakar     India 2019-03-29   
4718   s6530    Movie  Consequences        Ozan Açıktan    Turkey 2019-10-25   

      release_year rating  duration                                listed_in  \
393           2018  TV-14  1 Season        International TV Shows, TV Dramas   
537           2018  TV-14  1 Season        International TV Shows, TV Dramas   
2590          2014 

## Step 4: Standardize Columns
- **type:** consistent capitalization (Movie / TV Show)
- **date_added:** converted to datetime, with a new `year_added` column
- **duration:** split into `duration_value` (number) and `duration_unit` (min / Season)
- **rating:** grouped into audience categories (Kids, Teens, Adults, Unrated) in a new `audience` column

In [19]:
# Type
df["type"] = df["type"].str.title()

# Date
df["date_added"] = pd.to_datetime(df["date_added"], format="%m/%d/%Y", errors="coerce")
df["year_added"] = df["date_added"].dt.year

# Duration: split "90 min" / "2 Seasons" into number and unit
df["duration_value"] = df["duration"].str.extract(r"(\d+)")[0].astype(int)
df["duration_unit"] = df["duration"].str.extract(r"([A-Za-z]+)")[0].str.replace("Seasons", "Season")

# Rating: group into audience categories
rating_map = {
    "TV-Y": "Kids", "TV-Y7": "Kids", "TV-Y7-FV": "Kids", "TV-G": "Kids", "G": "Kids",
    "TV-PG": "Teens", "PG": "Teens", "TV-14": "Teens", "PG-13": "Teens",
    "TV-MA": "Adults", "R": "Adults", "NC-17": "Adults",
    "NR": "Unrated", "UR": "Unrated",
}
df["audience"] = df["rating"].map(rating_map)

df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,year_added,duration_value,duration_unit,audience
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,90,min,Teens
1,s3,Tv Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,1,Season,Adults
2,s6,Tv Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,1,Season,Adults
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,91,min,Teens
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,125,min,Adults


## Final Data Quality Check
Verify the data types, content type counts, audience categories and the final shape after cleaning.

In [9]:
print(df.dtypes)
print()
print(df["type"].value_counts())
print()
print(df["audience"].value_counts())
print()
print("Final shape:", df.shape)

show_id                      str
type                         str
title                        str
director                     str
country                      str
date_added        datetime64[us]
release_year               int64
rating                       str
duration                     str
listed_in                    str
year_added                 int32
duration_value             int32
duration_unit                str
audience                     str
dtype: object

type
Movie      6126
Tv Show    2664
Name: count, dtype: int64

audience
Adults     4007
Teens      3795
Kids        906
Unrated      82
Name: count, dtype: int64

Final shape: (8790, 14)


## Step 5: Export the Cleaned Dataset
Save the cleaned data as `netflix_clean.csv` for use in the next analysis tasks.

import os
os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/netflix_clean.csv", index=False)
print("Saved!")

## Summary of Cleaning Steps
- Converted hidden placeholders ("Not Given") to real missing values and filled them with "Unknown"
- Removed duplicate records and trimmed whitespace
- Converted `date_added` to datetime and created `year_added`
- Split `duration` into value and unit
- Grouped ratings into audience categories
- Exported the final dataset to `netflix_clean.csv`